In [16]:
from src.config import  ANALYTICAL_DATA_PATH
import pandas as pd
import matplotlib.pyplot as plt

In [30]:
df = pd.read_csv(ANALYTICAL_DATA_PATH / 'combined.csv', parse_dates=['period'])

df.head()

,Region Code,Provider Code,Provider Name,Treatment Function Code,Treatment Function,>0-1,>1-2,>2-3,>3-4,>4-5,...,92nd percentile waiting time (in weeks),Total 52 plus weeks,Total 78 plus weeks,Total 65 plus weeks,% 52 plus weeks,period,month_name,month,year,month_label
0,Y62,RXN,LANCASHIRE TEACHING HOSPITALS NHS FOUNDATION T...,C_110,Trauma and Orthopaedic Service,94,177,129,133,155,...,37.207273,14,0,0,0.003093,2026-04-01,Apr,4,2026,Apr 2026
1,Y62,RXN,LANCASHIRE TEACHING HOSPITALS NHS FOUNDATION T...,C_110,Trauma and Orthopaedic Service,168,176,132,181,224,...,37.470526,37,0,0,NaN,2025-02-01,Feb,2,2025,Feb 2025
2,Y62,RXN,LANCASHIRE TEACHING HOSPITALS NHS FOUNDATION T...,C_110,Trauma and Orthopaedic Service,124,238,147,183,110,...,43.987500,171,5,34,NaN,2023-06-01,Jun,6,2023,Jun 2023
3,Y62,RXN,LANCASHIRE TEACHING HOSPITALS NHS FOUNDATION T...,C_110,Trauma and Orthopaedic Service,153,210,173,157,159,...,43.695000,162,8,35,NaN,2023-07-01,Jul,7,2023,Jul 2023
4,Y62,RXN,LANCASHIRE TEACHING HOSPITALS NHS FOUNDATION T...,C_110,Trauma and Orthopaedic Service,48,203,193,98,111,...,37.841000,10,0,0,0.002586,2025-03-01,Mar,3,2025,Mar 2025


In [32]:
# the model is to predictthe total number of incomplete pathways each month

selected_cols = [
    'Total number of incomplete pathways',
    'period',
]

df_sf = df[selected_cols].copy()
df_sf.rename(
   columns= {
         'Total number of incomplete pathways': 'y',
         'period': 'ds'      
    },
   inplace=True
)
df_sf["unique_id"] = "RXN_C110"


In [38]:
df_sf.head()

df_sf.sort_values('ds')

,y,ds,unique_id
42,3301,2023-01-01,RXN_C110
16,3128,2023-02-01,RXN_C110
19,3072,2023-03-01,RXN_C110
23,3133,2023-04-01,RXN_C110
29,3268,2023-05-01,RXN_C110
2,3465,2023-06-01,RXN_C110
3,3679,2023-07-01,RXN_C110
12,3738,2023-08-01,RXN_C110
28,3903,2023-09-01,RXN_C110
39,3827,2023-10-01,RXN_C110


In [44]:
# split the dataset into training and testing data set, the training data set will comprise of observation from January 2023 to December 2025
training_data = df_sf[
    df_sf["ds"] <= "2025-12-01"
].copy()

test_data = df_sf[
    df_sf["ds"] >= "2026-01-01"
].copy()

horizon = len(test_data)

horizon

7

In [45]:
# create the baseline model
from statsforecast import StatsForecast
from statsforecast.models import (
    Naive,
    HistoricAverage,
    SeasonalNaive,
    WindowAverage
)

models = [
    Naive(),
    HistoricAverage(),
    SeasonalNaive(12),
    WindowAverage(3)
]


sf = StatsForecast(
    models=models,
    freq='MS'
)
sf.fit(df=training_data)

predictions = sf.predict(h=len(test_data))

predictions

,unique_id,ds,Naive,HistoricAverage,SeasonalNaive,WindowAverage
0,RXN_C110,2026-01-01,4450.0,3818.555664,4032.0,4400.666504
1,RXN_C110,2026-02-01,4450.0,3818.555664,4114.0,4400.666504
2,RXN_C110,2026-03-01,4450.0,3818.555664,3867.0,4400.666504
3,RXN_C110,2026-04-01,4450.0,3818.555664,3725.0,4400.666504
4,RXN_C110,2026-05-01,4450.0,3818.555664,3908.0,4400.666504
5,RXN_C110,2026-06-01,4450.0,3818.555664,4032.0,4400.666504
6,RXN_C110,2026-07-01,4450.0,3818.555664,4084.0,4400.666504
